In [ ]:
import math
import random
import csv
import json
from pathlib import Path
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm, PowerNorm, LinearSegmentedColormap


# ============================================================
# Mode
# ============================================================
# True  : run experiments and draw figures
# False : load saved npz file and redraw figures only

RUN_EXPERIMENT = True


# ============================================================
# Seed
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# ============================================================
# Config
# ============================================================

@dataclass
class Config:
    num_samples: int = 5000
    seq_len: int = 64
    dim: int = 24

    # Synthetic data generation (Eq. 138 in the paper)
    base_noise: float = 0.35                 # sigma_x
    relevant_signal_total: float = 3.0       # C
    distractor_label_noise: float = 1.0      # sigma_d
    relevance_direction_strength: float = 1.0  # alpha

    # grid
    relevant_counts: tuple = (1, 2, 4, 8, 16, 24, 32)
    score_noises: tuple = (0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.00)

    num_runs: int = 10
    base_seed: int = 10


# ============================================================
# Parameters for value aggregation potentials
# ============================================================

# Power potential:
#   psi_P(v) = (1/P) sum_k |v_k|^P
# P = 2 corresponds to the quadratic potential.
POWER_P = 3.5

# Cosh potential:
#   psi_beta(v) = (1/beta^2) sum_k {cosh(beta v_k) - 1}
COSH_BETA = 3.5

# Numerical clipping for sinh(beta x)
COSH_CLIP_VALUE = 20.0


# ============================================================
# Colormap settings
# ============================================================

MACARON_MIX_CMAP = LinearSegmentedColormap.from_list(
    "macaron_mix",
    [
        "#fffdfc",
        "#fbf6f8",
        "#f4eef7",
        "#ebeaf5",
        "#deebf0",
        "#d0e4e4",
    ]
)


# ============================================================
# Attention mappings
# ============================================================

def sparsemax(logits: torch.Tensor, dim: int = -1):
    z_sorted, _ = torch.sort(logits, descending=True, dim=dim)
    z_cumsum = torch.cumsum(z_sorted, dim=dim)

    r = torch.arange(
        1,
        logits.size(dim) + 1,
        device=logits.device,
        dtype=logits.dtype,
    )

    shape = [1] * logits.dim()
    shape[dim] = -1
    r = r.view(shape)

    support = 1 + r * z_sorted > z_cumsum
    k = support.sum(dim=dim, keepdim=True)

    tau = (torch.gather(z_cumsum, dim, k - 1) - 1) / k.to(logits.dtype)
    return torch.clamp(logits - tau, min=0.0)


def attention(scores: torch.Tensor, mapping: str):
    if mapping == "softmax":
        return F.softmax(scores, dim=-1)
    if mapping == "sparsemax":
        return sparsemax(scores, dim=-1)
    raise ValueError(mapping)


# ============================================================
# Data generation
# ============================================================

def normalize(v):
    norm = np.linalg.norm(v)
    if norm == 0.0:
        raise ValueError("Cannot normalize the zero vector.")
    return v / norm


def generate_orthonormal_unit_vectors(
    dim: int,
    rng: np.random.RandomState,
):
    """Generate unit vectors u and r with <u, r> = 0."""
    u = normalize(rng.randn(dim).astype(np.float64))

    while True:
        r = rng.randn(dim).astype(np.float64)
        r = r - np.dot(r, u) * u
        r_norm = np.linalg.norm(r)
        if r_norm > 1e-12:
            r = r / r_norm
            break

    return u.astype(np.float32), r.astype(np.float32)


def generate_data(
    cfg: Config,
    num_relevant: int,
    score_noise: float,
    seed: int,
):
    """
    Generate the synthetic data exactly according to Eqs. (138)--(141).

    x_j = xi_j + epsilon_j u
          + 1[j in R] (C / sqrt(m)) y u
          + 1[j in R] alpha r,

    s_j = <x_j, r> + eta_j,

    where ||u|| = ||r|| = 1 and <u, r> = 0.
    """
    rng = np.random.RandomState(seed)

    N = cfg.num_samples
    n = cfg.seq_len
    d = cfg.dim
    m = num_relevant

    u, r = generate_orthonormal_unit_vectors(d, rng)

    xs = []
    scores = []
    ys = []
    rel_masks = []

    for _ in range(N):
        y = rng.choice([-1.0, 1.0])

        # Background noise xi_j ~ N(0, sigma_x^2 I_d)
        x = rng.randn(n, d).astype(np.float32) * cfg.base_noise
        rel_mask = np.zeros(n, dtype=np.float32)

        R = rng.choice(n, size=m, replace=False)
        rel_mask[R] = 1.0

        # Label-direction noise epsilon_j u, epsilon_j ~ N(0, sigma_d^2)
        label_direction_noise = (
            rng.randn(n).astype(np.float32)
            * cfg.distractor_label_noise
        )
        x += label_direction_noise[:, None] * u[None, :]

        # Label signal 1[j in R] (C / sqrt(m)) y u
        per_token_signal = cfg.relevant_signal_total / math.sqrt(m)
        x[R] += per_token_signal * y * u[None, :]

        # Relevance component 1[j in R] alpha r
        x[R] += cfg.relevance_direction_strength * r[None, :]

        # Inner-product attention score with fixed query q = r:
        # s_j = <x_j, r> + eta_j, eta_j ~ N(0, sigma_s^2).
        eta = rng.randn(n).astype(np.float32) * score_noise
        s = x @ r + eta

        xs.append(x)
        scores.append(s.astype(np.float32))
        ys.append(1.0 if y > 0 else 0.0)
        rel_masks.append(rel_mask)

    return {
        "x": torch.tensor(np.stack(xs), dtype=torch.float32),
        "scores": torch.tensor(np.stack(scores), dtype=torch.float32),
        "y": torch.tensor(np.array(ys), dtype=torch.float32),
        "rel_mask": torch.tensor(np.stack(rel_masks), dtype=torch.float32),
        "u": torch.tensor(u, dtype=torch.float32),
        "r": torch.tensor(r, dtype=torch.float32),
    }


# ============================================================
# Value aggregation rules
# ============================================================

def aggregate_linear(x: torch.Tensor, a: torch.Tensor):
    """
    Standard linear weighted aggregation.
    """
    return torch.sum(a.unsqueeze(-1) * x, dim=1)


def aggregate_power(
    x: torch.Tensor,
    a: torch.Tensor,
    P: float = 3.5,
):
    """
    Power-potential value aggregation from Eq. (204).
    """
    if P <= 1.0:
        raise ValueError("POWER_P must be greater than 1.")

    grad_x = torch.sign(x) * torch.pow(torch.abs(x), P - 1.0)
    mean_dual = torch.sum(a.unsqueeze(-1) * grad_x, dim=1)

    z = torch.sign(mean_dual) * torch.pow(
        torch.abs(mean_dual),
        1.0 / (P - 1.0),
    )
    return z


def aggregate_cosh(
    x: torch.Tensor,
    a: torch.Tensor,
    beta: float = 3.5,
    clip_value: float = 20.0,
):
    """
    Hyperbolic-cosine value aggregation.
    """
    if beta <= 0.0:
        raise ValueError("COSH_BETA must be positive.")

    bx = beta * x

    if clip_value is not None:
        bx = torch.clamp(bx, -clip_value, clip_value)

    mean_dual = torch.sum(a.unsqueeze(-1) * torch.sinh(bx), dim=1)
    z = torch.asinh(mean_dual) / beta

    return z


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate_all(data):
    x = data["x"]
    scores = data["scores"]
    y = data["y"]
    rel_mask = data["rel_mask"]
    u = data["u"]

    # --------------------------------------------------------
    # Attention weights
    # --------------------------------------------------------

    a_soft = attention(scores, "softmax")
    a_sparse = attention(scores, "sparsemax")

    # --------------------------------------------------------
    # Standard:
    #   softmax attention weights + linear aggregation
    # --------------------------------------------------------

    z_standard = aggregate_linear(x, a_soft)
    logit_standard = torch.matmul(z_standard, u)
    pred_standard = (logit_standard >= 0).float()
    acc_standard = (pred_standard == y).float().mean().item()

    # --------------------------------------------------------
    # Condition 1:
    #   sparsemax attention weights + linear aggregation
    # --------------------------------------------------------

    z_sparse = aggregate_linear(x, a_sparse)
    logit_sparse = torch.matmul(z_sparse, u)
    pred_sparse = (logit_sparse >= 0).float()
    acc_sparse = (pred_sparse == y).float().mean().item()

    # --------------------------------------------------------
    # Condition 2:
    #   softmax attention weights + power aggregation
    # --------------------------------------------------------

    z_power = aggregate_power(x, a_soft, P=POWER_P)
    logit_power = torch.matmul(z_power, u)
    pred_power = (logit_power >= 0).float()
    acc_power = (pred_power == y).float().mean().item()

    # --------------------------------------------------------
    # Condition 3:
    #   softmax attention weights + cosh aggregation
    # --------------------------------------------------------

    z_cosh = aggregate_cosh(
        x,
        a_soft,
        beta=COSH_BETA,
        clip_value=COSH_CLIP_VALUE,
    )
    logit_cosh = torch.matmul(z_cosh, u)
    pred_cosh = (logit_cosh >= 0).float()
    acc_cosh = (pred_cosh == y).float().mean().item()

    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------

    eps = 1e-12

    soft_rel_mass = (a_soft * rel_mask).sum(dim=-1).mean().item()
    sparse_rel_mass = (a_sparse * rel_mask).sum(dim=-1).mean().item()

    soft_concentration = torch.sum(a_soft ** 2, dim=-1).mean().item()
    sparse_concentration = torch.sum(a_sparse ** 2, dim=-1).mean().item()

    soft_entropy = -(
        a_soft.clamp_min(eps) * torch.log(a_soft.clamp_min(eps))
    ).sum(dim=-1).mean().item()

    sparse_entropy = -(
        a_sparse.clamp_min(eps) * torch.log(a_sparse.clamp_min(eps))
    ).sum(dim=-1).mean().item()

    return {
        # Accuracies
        "acc_standard": acc_standard,
        "acc_sparse": acc_sparse,
        "acc_power": acc_power,
        "acc_cosh": acc_cosh,

        # Differences from standard
        "diff_sparse": acc_sparse - acc_standard,
        "diff_power": acc_power - acc_standard,
        "diff_cosh": acc_cosh - acc_standard,

        # Diagnostics
        "soft_rel_mass": soft_rel_mass,
        "sparse_rel_mass": sparse_rel_mass,
        "soft_concentration": soft_concentration,
        "sparse_concentration": sparse_concentration,
        "soft_entropy": soft_entropy,
        "sparse_entropy": sparse_entropy,
    }


# ============================================================
# Grid experiment
# ============================================================

def get_result_keys():
    return [
        "acc_standard",
        "acc_sparse",
        "acc_power",
        "acc_cosh",
        "diff_sparse",
        "diff_power",
        "diff_cosh",
        "soft_rel_mass",
        "sparse_rel_mass",
        "soft_concentration",
        "sparse_concentration",
        "soft_entropy",
        "sparse_entropy",
    ]


def run_grid(cfg: Config):
    M = len(cfg.relevant_counts)
    S = len(cfg.score_noises)
    keys = get_result_keys()

    results = {
        "mean": {
            key: np.zeros((M, S), dtype=np.float64)
            for key in keys
        },
        "std": {
            key: np.zeros((M, S), dtype=np.float64)
            for key in keys
        },
    }

    raw_rows = []
    summary_rows = []

    for i, m in enumerate(cfg.relevant_counts):
        for j, sigma in enumerate(cfg.score_noises):
            vals = defaultdict(list)

            for run in range(cfg.num_runs):
                seed = cfg.base_seed + 1000 * i + 100 * j + run

                data = generate_data(
                    cfg=cfg,
                    num_relevant=m,
                    score_noise=sigma,
                    seed=seed,
                )

                out = evaluate_all(data)

                raw_row = {
                    "num_relevant": m,
                    "score_noise": sigma,
                    "run": run,
                    "seed": seed,
                }

                for key, value in out.items():
                    vals[key].append(value)
                    raw_row[key] = value

                raw_rows.append(raw_row)

            summary_row = {
                "num_relevant": m,
                "score_noise": sigma,
            }

            for key in keys:
                arr = np.asarray(vals[key], dtype=np.float64)

                mean_value = np.mean(arr)

                if cfg.num_runs >= 2:
                    std_value = np.std(arr, ddof=1)
                else:
                    std_value = 0.0

                results["mean"][key][i, j] = mean_value
                results["std"][key][i, j] = std_value

                summary_row[f"{key}_mean"] = mean_value
                summary_row[f"{key}_std"] = std_value

            summary_rows.append(summary_row)

            print(
                f"m={m:2d}, score_noise={sigma:.2f} | "
                f"standard={results['mean']['acc_standard'][i, j]:.4f}"
                f"±{results['std']['acc_standard'][i, j]:.4f}, "
                f"sparse={results['mean']['acc_sparse'][i, j]:.4f}"
                f"±{results['std']['acc_sparse'][i, j]:.4f}, "
                f"power={results['mean']['acc_power'][i, j]:.4f}"
                f"±{results['std']['acc_power'][i, j]:.4f}, "
                f"cosh={results['mean']['acc_cosh'][i, j]:.4f}"
                f"±{results['std']['acc_cosh'][i, j]:.4f}, "
                f"d_sparse={results['mean']['diff_sparse'][i, j]:+.4f}"
                f"±{results['std']['diff_sparse'][i, j]:.4f}, "
                f"d_power={results['mean']['diff_power'][i, j]:+.4f}"
                f"±{results['std']['diff_power'][i, j]:.4f}, "
                f"d_cosh={results['mean']['diff_cosh'][i, j]:+.4f}"
                f"±{results['std']['diff_cosh'][i, j]:.4f}"
            )

    return results, raw_rows, summary_rows, keys


# ============================================================
# Saving and loading logs
# ============================================================

def save_experiment_logs(
    results,
    raw_rows,
    summary_rows,
    keys,
    cfg: Config,
    output_dir: str = "experiment_logs",
):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # --------------------------------------------------------
    # Save raw run-level data
    # --------------------------------------------------------

    raw_csv_path = output_dir / "raw_results.csv"

    if len(raw_rows) > 0:
        raw_fieldnames = list(raw_rows[0].keys())

        with open(raw_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=raw_fieldnames)
            writer.writeheader()
            writer.writerows(raw_rows)

    # --------------------------------------------------------
    # Save summary data
    # --------------------------------------------------------

    summary_csv_path = output_dir / "summary_results.csv"

    if len(summary_rows) > 0:
        summary_fieldnames = list(summary_rows[0].keys())

        with open(summary_csv_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=summary_fieldnames)
            writer.writeheader()
            writer.writerows(summary_rows)

    # --------------------------------------------------------
    # Save matrices for easy re-plotting
    # --------------------------------------------------------

    npz_dict = {}

    for key in keys:
        npz_dict[f"{key}_mean"] = results["mean"][key]
        npz_dict[f"{key}_std"] = results["std"][key]

    npz_dict["relevant_counts"] = np.asarray(cfg.relevant_counts)
    npz_dict["score_noises"] = np.asarray(cfg.score_noises)

    np.savez(output_dir / "results_summary.npz", **npz_dict)

    # --------------------------------------------------------
    # Save config
    # --------------------------------------------------------

    config_dict = {
        "num_samples": cfg.num_samples,
        "seq_len": cfg.seq_len,
        "dim": cfg.dim,
        "base_noise": cfg.base_noise,
        "relevant_signal_total": cfg.relevant_signal_total,
        "distractor_label_noise": cfg.distractor_label_noise,
        "relevance_direction_strength": cfg.relevance_direction_strength,
        "relevant_counts": list(cfg.relevant_counts),
        "score_noises": list(cfg.score_noises),
        "num_runs": cfg.num_runs,
        "base_seed": cfg.base_seed,
        "POWER_P": POWER_P,
        "COSH_BETA": COSH_BETA,
        "COSH_CLIP_VALUE": COSH_CLIP_VALUE,
    }

    with open(output_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(config_dict, f, indent=2)

    print("\nSaved logs:")
    print(f"  {raw_csv_path}")
    print(f"  {summary_csv_path}")
    print(f"  {output_dir / 'results_summary.npz'}")
    print(f"  {output_dir / 'config.json'}")


def load_results_from_npz(npz_path: str = "experiment_logs/results_summary.npz"):
    data = np.load(npz_path)

    relevant_counts = tuple(data["relevant_counts"].tolist())
    score_noises = tuple(data["score_noises"].tolist())

    cfg = Config(
        relevant_counts=relevant_counts,
        score_noises=score_noises,
    )

    results = {
        "mean": {},
        "std": {},
    }

    keys = []

    for name in data.files:
        if name.endswith("_mean"):
            key = name[:-5]
            keys.append(key)
            results["mean"][key] = data[name]

    for name in data.files:
        if name.endswith("_std"):
            key = name[:-4]
            results["std"][key] = data[name]

    return results, cfg, keys


# ============================================================
# Plotting
# ============================================================

def plot_heatmap(
    mean_matrix,
    cfg: Config,
    title: str,
    filename,
    cmap,
    norm=None,
    std_matrix=None,
    mean_format="{:.3f}",
    std_format="{:.3f}",
    colorbar_label=None,
):
    fig, ax = plt.subplots(figsize=(8.8, 5.4))

    im = ax.imshow(
        mean_matrix,
        aspect="auto",
        origin="lower",
        cmap=cmap,
        norm=norm,
    )

    ax.set_xticks(np.arange(len(cfg.score_noises)))
    ax.set_xticklabels([str(x) for x in cfg.score_noises])

    ax.set_yticks(np.arange(len(cfg.relevant_counts)))
    ax.set_yticklabels([str(x) for x in cfg.relevant_counts])

    ax.set_xlabel(r"score noise $\sigma_s$")
    ax.set_ylabel(r"number of relevant tokens $m$")
    ax.set_title(title)

    for i in range(mean_matrix.shape[0]):
        for j in range(mean_matrix.shape[1]):
            mean_text = mean_format.format(mean_matrix[i, j])

            if std_matrix is not None:
                std_text = std_format.format(std_matrix[i, j])
                text = f"{mean_text}\n±{std_text}"
            else:
                text = mean_text

            ax.text(
                j,
                i,
                text,
                ha="center",
                va="center",
                fontsize=7,
                color="black",
            )

    cbar = fig.colorbar(im, ax=ax)
    if colorbar_label is not None:
        cbar.set_label(colorbar_label)

    fig.tight_layout()
    fig.savefig(filename, dpi=300)
    plt.close(fig)


def plot_all(results, cfg: Config, output_dir: str = "."):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    mean = results["mean"]
    std = results["std"]

    # --------------------------------------------------------
    # Accuracy heatmaps
    # --------------------------------------------------------

    acc_norm = PowerNorm(
        gamma=1.0,
        vmin=0.5,
        vmax=1.0,
    )

    plot_heatmap(
        mean["acc_standard"],
        cfg,
        title="Accuracy: Standard Setting",
        filename=output_dir / "accuracy_standard_softmax_linear.png",
        cmap=MACARON_MIX_CMAP,
        norm=acc_norm,
        std_matrix=std["acc_standard"],
        mean_format="{:.3f}",
        std_format="{:.3f}",
        colorbar_label="Accuracy",
    )

    plot_heatmap(
        mean["acc_sparse"],
        cfg,
        title="Accuracy: Sparsemax + Linear Aggregation",
        filename=output_dir / "accuracy_sparsemax_linear.png",
        cmap=MACARON_MIX_CMAP,
        norm=acc_norm,
        std_matrix=std["acc_sparse"],
        mean_format="{:.3f}",
        std_format="{:.3f}",
        colorbar_label="Accuracy",
    )

    plot_heatmap(
        mean["acc_power"],
        cfg,
        title=rf"Accuracy: Softmax + Power Aggregation $(P={POWER_P})$",
        filename=output_dir / "accuracy_softmax_power.png",
        cmap=MACARON_MIX_CMAP,
        norm=acc_norm,
        std_matrix=std["acc_power"],
        mean_format="{:.3f}",
        std_format="{:.3f}",
        colorbar_label="Accuracy",
    )

    plot_heatmap(
        mean["acc_cosh"],
        cfg,
        title=rf"Accuracy: Softmax + Cosh Aggregation $(\beta={COSH_BETA})$",
        filename=output_dir / "accuracy_softmax_cosh.png",
        cmap=MACARON_MIX_CMAP,
        norm=acc_norm,
        std_matrix=std["acc_cosh"],
        mean_format="{:.3f}",
        std_format="{:.3f}",
        colorbar_label="Accuracy",
    )

    # --------------------------------------------------------
    # Difference heatmaps from the standard setting
    # --------------------------------------------------------

    max_abs_delta = max(
        np.max(np.abs(mean["diff_sparse"])),
        np.max(np.abs(mean["diff_power"])),
        np.max(np.abs(mean["diff_cosh"])),
        1e-6,
    )

    delta_norm = TwoSlopeNorm(
        vmin=-max_abs_delta,
        vcenter=0.0,
        vmax=max_abs_delta,
    )

    plot_heatmap(
        mean["diff_sparse"],
        cfg,
        title="Accuracy Difference: Sparsemax + Linear - Standard",
        filename=output_dir / "diff_sparsemax_linear_minus_standard.png",
        cmap="coolwarm",
        norm=delta_norm,
        std_matrix=std["diff_sparse"],
        mean_format="{:+.3f}",
        std_format="{:.3f}",
        colorbar_label="Accuracy difference from standard",
    )

    plot_heatmap(
        mean["diff_power"],
        cfg,
        title=rf"Accuracy Difference: Softmax + Power$(P={POWER_P})$ - Standard",
        filename=output_dir / "diff_softmax_power_minus_standard.png",
        cmap="coolwarm",
        norm=delta_norm,
        std_matrix=std["diff_power"],
        mean_format="{:+.3f}",
        std_format="{:.3f}",
        colorbar_label="Accuracy difference from standard",
    )

    plot_heatmap(
        mean["diff_cosh"],
        cfg,
        title=rf"Accuracy Difference: Softmax + Cosh$(\beta={COSH_BETA})$ - Standard",
        filename=output_dir / "diff_softmax_cosh_minus_standard.png",
        cmap="coolwarm",
        norm=delta_norm,
        std_matrix=std["diff_cosh"],
        mean_format="{:+.3f}",
        std_format="{:.3f}",
        colorbar_label="Accuracy difference from standard",
    )

    print("\nSaved figures:")
    print(f"  {output_dir / 'accuracy_standard_softmax_linear.png'}")
    print(f"  {output_dir / 'accuracy_sparsemax_linear.png'}")
    print(f"  {output_dir / 'accuracy_softmax_power.png'}")
    print(f"  {output_dir / 'accuracy_softmax_cosh.png'}")
    print(f"  {output_dir / 'diff_sparsemax_linear_minus_standard.png'}")
    print(f"  {output_dir / 'diff_softmax_power_minus_standard.png'}")
    print(f"  {output_dir / 'diff_softmax_cosh_minus_standard.png'}")


# ============================================================
# Main
# ============================================================

def main():
    set_seed(10)

    output_dir = "experiment_logs"

    cfg = Config(
        num_samples=5000,
        seq_len=64,
        dim=24,

        base_noise=0.35,
        relevant_signal_total=3.0,
        distractor_label_noise=1.0,
        relevance_direction_strength=1.0,

        relevant_counts=(1, 2, 4, 8, 16, 24, 32),
        score_noises=(0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.00),

        num_runs=10,
        base_seed=10,
    )

    if RUN_EXPERIMENT:
        results, raw_rows, summary_rows, keys = run_grid(cfg)

        save_experiment_logs(
            results=results,
            raw_rows=raw_rows,
            summary_rows=summary_rows,
            keys=keys,
            cfg=cfg,
            output_dir=output_dir,
        )

        plot_all(results, cfg, output_dir=output_dir)

    else:
        results, cfg, keys = load_results_from_npz(
            "experiment_logs/results_summary.npz"
        )

        plot_all(
            results,
            cfg,
            output_dir="experiment_logs_replot",
        )


if __name__ == "__main__":
    main()